In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

DATA    = Path.cwd().parent / 'data' / 'interim'
FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

BLUE, RED, GREEN, GRAY = '#2563EB','#DC2626','#16A34A','#6B7280'
THRESH = 0.5
IR     = 0.10   # intervention response rate (Ascarza 2018)

features = pd.read_csv(DATA / 'features.csv').drop_duplicates(subset='client_id', keep='first')
preds    = pd.read_csv(DATA / 'test_predictions.csv')

# 90-day test set with feature columns
test_full = features.merge(preds[['client_id','proba_rf','proba_xgb','proba_lr']], on='client_id', how='inner')
print(f'features: {features.shape}  preds: {preds.shape}  test_full: {test_full.shape}')
assert len(test_full) == len(preds), 'Size mismatch!'


In [ ]:
# 90-day model AUC comparison
y_true = preds['churned']
print(f"{'Model':<22} {'AUC-ROC':>9} {'Avg Prec':>10}")
print('-' * 44)
for name, col in [('Random Forest','proba_rf'),('XGBoost','proba_xgb'),('Logistic Reg.','proba_lr')]:
    auc = roc_auc_score(y_true, preds[col])
    ap  = average_precision_score(y_true, preds[col])
    print(f'  {name:<20} {auc:>9.4f} {ap:>10.4f}')


In [ ]:
# Targeting quality: multi-visit segment (tx>1)
target        = test_full[test_full['tx_count'] > 1].copy()
n_target      = len(target)
n_fading      = (target['churned'] == 1).sum()
base_rate     = n_fading / n_target

# Current system proxy: clients who received retention_offer messages
current   = target[target['msg_seg_retention_offer'] > 0]
cur_n     = len(current)
cur_tp    = (current['churned'] == 1).sum()
cur_prec  = cur_tp / cur_n
cur_recall= cur_tp / n_fading

# 90-day RF model at threshold 0.5
flagged   = target[target['proba_rf'] >= THRESH]
mod_n     = len(flagged)
mod_tp    = (flagged['churned'] == 1).sum()
mod_prec  = mod_tp / mod_n
mod_recall= mod_tp / n_fading

print(f'Target segment (tx>1): {n_target:,}  base churn rate: {base_rate*100:.1f}%')
print()
print(f"{'':35s} {'Current':>12} {'RF 90-day':>12}")
print(f"{'Messages sent':35s} {cur_n:>12,} {mod_n:>12,}")
print(f"{'TP (true churners):':35s} {cur_tp:>12,} {mod_tp:>12,}")
print(f"{'Precision':35s} {cur_prec*100:>11.1f}% {mod_prec*100:>11.1f}%")
print(f"{'Recall':35s} {cur_recall*100:>11.1f}% {mod_recall*100:>11.1f}%")
print(f"{'Lift over base rate':35s} {cur_prec/base_rate:>11.2f}x {mod_prec/base_rate:>11.2f}x")
print(f"{'FP per 1,000 messages':35s} {(1-cur_prec)*1000:>11.0f} {(1-mod_prec)*1000:>11.0f}")


In [ ]:
# Retention simulation (scale test set -> full eligible population)
# Uses prod model (NB09 mv test) -- loaded separately
preds_mv = pd.read_csv(DATA / 'test_predictions_mv.csv')

# Full multi-visit eligible population size
features_full = pd.read_csv(DATA / 'features.csv')
n_mv_full = len(features_full[features_full['tx_count'] > 1])
n_mv_test = len(preds_mv)
SCALE = n_mv_full / n_mv_test

# Prod model metrics on mv test
y_mv      = preds_mv['churned']
proba_mv  = preds_mv['proba_rf_mv']
flagged_mv = proba_mv >= THRESH

base_rate_mv = y_mv.mean()
mod_tp_mv    = (flagged_mv & (y_mv == 1)).sum()
mod_prec_mv  = mod_tp_mv / flagged_mv.sum()

# Current system proxy: mv test clients who received retention_offer messages
# Join features (for msg_seg_retention_offer) with ground-truth churned from preds_mv
features_mv_sub = features_full[
    features_full['client_id'].isin(preds_mv['client_id'])
][['client_id', 'msg_seg_retention_offer']].copy()
cur_mv = features_mv_sub.merge(preds_mv[['client_id', 'churned']], on='client_id', how='inner')
cur_mv_flagged = cur_mv[cur_mv['msg_seg_retention_offer'] > 0]
cur_tp_mv   = (cur_mv_flagged['churned'] == 1).sum()
cur_prec_mv = cur_tp_mv / len(cur_mv_flagged)

# Scale to full population
n_retained_full = int((1 - base_rate_mv) * n_mv_full)
cur_saved = int(cur_tp_mv * SCALE * IR)
mod_saved = int(mod_tp_mv * SCALE * IR)

def ret_rate(saved): return (n_retained_full + saved) / n_mv_full * 100

print(f'Scale: {SCALE:.2f}x  (mv full={n_mv_full:,}  mv test={n_mv_test:,})')
print(f'Baseline retention rate: {ret_rate(0):.2f}%  ({n_retained_full:,} retained)')
print()
print(f"{'Scenario':25s} {'Saved':>10} {'Ret.Rate':>10} {'Delta pp':>10}")
print('-' * 58)
print(f"{'Baseline':25s} {0:>10,} {ret_rate(0):>10.2f}% {0:>+9.2f}")
print(f"{'Current system':25s} {cur_saved:>10,} {ret_rate(cur_saved):>10.2f}% {ret_rate(cur_saved)-ret_rate(0):>+9.2f}")
print(f"{'Prod model (30d mv RF)':25s} {mod_saved:>10,} {ret_rate(mod_saved):>10.2f}% {ret_rate(mod_saved)-ret_rate(0):>+9.2f}")
print(f"\nDelta model vs current: +{mod_saved-cur_saved:,} clients  "
      f"+{ret_rate(mod_saved)-ret_rate(cur_saved):.2f} pp")


In [ ]:
# Sensitivity: effect of timing uplift on retention rate
timing_gains = [0.0, 0.10, 0.20, 0.30]
print(f"{'Timing uplift':25s} {'Eff. IR':>9} {'Saved':>10} {'Ret.Rate':>10} {'vs model':>10}")
print('-' * 67)
baseline_rr = ret_rate(mod_saved)
for g in timing_gains:
    ir_eff = IR * (1 + g)
    saved  = int(mod_tp_mv * SCALE * ir_eff)
    rr     = ret_rate(saved)
    label  = 'base (proven)' if g == 0 else f'+{g*100:.0f}% from timing'
    print(f'{label:25s} {ir_eff*100:>9.1f}% {saved:>10,} {rr:>10.2f}% {rr-baseline_rr:>+9.2f}')
